In [1]:
sidechain_weights = {
    (0,1): -0.002508, (0,2): -0.002910, (0,3):  0.003311, (0,4):  0.003738, (0,5): -0.002279,
    (0,6): -0.007833, (0,7):  0.005194, (0,8):  0.003015, (0,9): -0.001686,
    (1,2):  0.087800, (1,3): -0.012036, (1,4): -0.076552, (1,5): -0.006233, (1,6): -0.047745,
    (1,7):  0.048222, (1,8): -0.001081, (1,9): -0.107155,
    (2,3): -0.040101, (2,4):  0.005867, (2,5): -0.048390, (2,6):  0.022192, (2,7):  0.004446,
    (2,8): -0.119810, (2,9):  0.005866,
    (3,4): -0.013178, (3,5):  0.058196, (3,6):  0.033062, (3,7): -0.066796, (3,8):  0.002742,
    (3,9):  0.081948,
    (4,5): -0.026415, (4,6):  0.000497, (4,7):  0.031807, (4,8):  0.357662, (4,9): -0.045548,
    (5,6): -0.056291, (5,7): -0.007534, (5,8):  0.030413, (5,9): -0.001797,
    (6,7):  0.023387, (6,8): -0.006023, (6,9):  0.027803,
    (7,8): -0.012528, (7,9):  0.006922,
    (8,9):  0.023446,
}

backbone_weights = {
    (0,3): -0.253690, (0,4):  0.362085, (0,5): -0.162276, (0,8):  0.141739, (0,9): -0.070355,
    (1,4): -0.093817, (1,7):  0.223309, (1,8): -0.203515, (1,9):  0.060981,
    (2,5):  0.063055, (2,6):  0.210065, (2,7): -0.298718, (2,8):  0.076177, (2,9):  0.230413,
    (3,6): -0.121888, (3,7):  0.036819, (3,8):  0.181392, (3,9):  0.070496,
    (4,7):  0.170268, (4,8): -0.080730, (4,9): -0.362873,
    (5,8): -0.009312, (5,9):  0.140581,
    (6,9): -0.096582,
}

assert max(max(i, j) for d in (sidechain_weights, backbone_weights) for (i, j) in d) <= 9


In [ ]:
import numpy as np
import pandas as pd

N = max(max(i,j) for d in (sidechain_weights, backbone_weights) for (i,j) in d) + 1

def per_residue(d):
    rows = []
    for (i,j), w in d.items():
        rows.append((i, abs(w), w))
        rows.append((j, abs(w), w))
    df = pd.DataFrame(rows, columns=["r","absw","w"])
    agg = df.groupby("r").agg(abs_sum=("absw","sum"), signed_sum=("w","sum"), degree=("r","size"))
    agg["mean_abs"] = agg["abs_sum"] / agg["degree"]
    return agg

side = per_residue(sidechain_weights).add_prefix("side_")
back = per_residue(backbone_weights).add_prefix("back_")

res = pd.DataFrame(index=range(N)).join(side, how="left").join(back, how="left").fillna(0.0)
res["total_abs"] = res["side_abs_sum"] + res["back_abs_sum"]
res["total_signed"] = res["side_signed_sum"] + res["back_signed_sum"]
res["degree"] = res["side_degree"] + res["back_degree"]
res["mean_abs"] = res["total_abs"] / res["degree"].replace(0, np.nan)
res["side_mean_abs"] = res["side_abs_sum"] / res["side_degree"].replace(0, np.nan)
res["back_mean_abs"] = res["back_abs_sum"] / res["back_degree"].replace(0, np.nan)

def z(x):
    x = x.copy()
    m, s = np.nanmean(x), np.nanstd(x)
    if s == 0: return x*0
    return (x - m) / s

res["z_total_abs"] = z(res["total_abs"])
res["z_mean_abs"] = z(res["mean_abs"])
res["hybrid_score"] = res["z_total_abs"] + res["z_mean_abs"]

rank_total = res.sort_values("total_abs", ascending=False)
rank_mean = res.sort_values("mean_abs", ascending=False)
rank_hybrid = res.sort_values("hybrid_score", ascending=False)
rank_side = res.sort_values("side_abs_sum", ascending=False)
rank_back = res.sort_values("back_abs_sum", ascending=False)

print("Top by total |w|:")
print(rank_total[["total_abs","side_abs_sum","back_abs_sum"]].round(6).head(10))
print("\nTop by mean |w|:")
print(rank_mean[["mean_abs"]].round(4).head(10))
print("\nTop by hybrid_score (z_total_abs + z_mean_abs):")
print(rank_hybrid[["hybrid_score","total_abs","mean_abs"]].round(6).head(10))
print("\nTop by sidechain |w|:")
print(rank_side[["side_abs_sum","side_mean_abs"]].round(6).head(10))
print("\nTop by backbone |w|:")
print(rank_back[["back_abs_sum","back_mean_abs"]].round(6).head(10))

df_edges = pd.concat([
    pd.DataFrame([(i,j,w,"side") for (i,j),w in sidechain_weights.items()], columns=["i","j","w","type"]),
    pd.DataFrame([(i,j,w,"back") for (i,j),w in backbone_weights.items()], columns=["i","j","w","type"])
], ignore_index=True)
print("\nTop individual edges by |w|:")
print(df_edges.assign(absw=df_edges.w.abs()).sort_values("absw", ascending=False).head(10)[["type","i","j","w"]])

best_overall = int(rank_hybrid.index[0])
best_total = int(rank_total.index[0])
best_mean = int(rank_mean.index[0])
print(f"\nMost important (hybrid): {best_overall}")
print(f"Most connected-by-strength (total |w|): {best_total}")
print(f"Strongest-per-connection (mean |w|): {best_mean}")

Top by total |w|:
   total_abs  side_abs_sum  back_abs_sum
4   1.631037      0.561264      1.069773
9   1.334452      0.302171      1.032281
8   1.249585      0.556720      0.692865
2   1.215810      0.337382      0.878428
0   1.022619      0.032474      0.990145
3   0.975655      0.311370      0.664285
1   0.970954      0.389332      0.581622
7   0.935950      0.206836      0.729114
6   0.653368      0.224833      0.428535
5   0.612772      0.237548      0.375224

Top by mean |w|:
   mean_abs
4     0.117
2     0.087
9     0.083
8     0.083
1     0.075
0     0.073
7     0.072
3     0.070
6     0.054
5     0.047

Top by hybrid_score (z_total_abs + z_mean_abs):
   hybrid_score  total_abs  mean_abs
4      4.195533   1.631037  0.116503
9      1.341806   1.334452  0.083403
2      1.128385   1.215810  0.086844
8      1.046784   1.249585  0.083306
0     -0.298655   1.022619  0.073044
1     -0.383447   0.970954  0.074689
3     -0.645562   0.975655  0.069690
7     -0.652712   0.935950  0.071996